In [1]:
# ─────────────────────────────────────────────────────
# P3 – CINEMATICA INVERSA 
# ─────────────────────────────────────────────────────
import sys
import os
import numpy as np
import pandas as pd
import time
import math
from typing import List

# Agregar rutas de módulos P1 y P2
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'P1_DH'))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'P2_FK'))

print('Librerías cargadas ✓')

# ─────────────────────────────────────────────────────
# CONEXION AL ROBOT
# ─────────────────────────────────────────────────────
from pymycobot.mycobot import MyCobot

mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)
assert mc.is_controller_connected(), 'Error de conexión'
print('Robot conectado ✓')


Librerías cargadas ✓
Robot conectado ✓


## Importar módulos de cinemática P1, P2 y P3

In [2]:
# Importar módulos de P1, P2 y P3
try:
    from p1_dh_representacion import DH_TABLE, forward_kinematics, extract_position
    from p2_cinematica_directa import ForwardKinematics
    from p3_cinematica_inversa import InverseKinematics, compare_ik_solutions
    print('Módulos P1, P2, P3 importados ✓')
except ImportError as e:
    print(f'[ERROR] No se pudieron importar módulos: {e}')
    raise

# ─────────────────────────────────────────────────────
# TABLA DE PARAMETROS DH (referencia de P1)
# ─────────────────────────────────────────────────────
DH_TABLE_LOCAL = [
    (   0,   134.75,  90.0,   0.0),   # J1 – Base
    (-110,     0.0,   0.0,  -90.0),   # J2 – Hombro
    ( -96,     0.0,   0.0,    0.0),   # J3 – Codo
    (   0,    63.40,  90.0,  -90.0),  # J4 – Muñeca 1
    (   0,    75.05, -90.0,   90.0),  # J5 – Muñeca 2
    (   0,    50.00,   0.0,    0.0),  # J6 – Gripper
]

JOINT_NAMES = ['J1 Base', 'J2 Hombro', 'J3 Codo', 'J4 Muñeca1', 'J5 Muñeca2', 'J6 Gripper']
JOINT_LIMITS = [
    (-168, 168), (-135, 90), (-150, 150),
    (-145, 145), (-165, 165), (-180, 180),
]

# ─────────────────────────────────────────────────────
# Funciones de Cinemática Directa (FK)
# ─────────────────────────────────────────────────────
def dh_matrix(theta_deg, d, a, alpha_deg):
    """Matriz de transformación homogénea DH 4x4 para un eslabón."""
    from math import radians, cos, sin
    theta = radians(theta_deg)
    alpha = radians(alpha_deg)
    ct, st = cos(theta), sin(theta)
    ca, sa = cos(alpha), sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [ 0,     sa,     ca,    d],
        [ 0,      0,      0,    1],
    ], dtype=float)

def build_Ti_list(thetas_deg):
    """Devuelve lista de 6 matrices Ti para los ángulos dados."""
    return [
        dh_matrix(thetas_deg[i] + DH_TABLE_LOCAL[i][3],
                  DH_TABLE_LOCAL[i][1],
                  DH_TABLE_LOCAL[i][0],
                  DH_TABLE_LOCAL[i][2])
        for i in range(6)
    ]

def forward_kinematics(thetas_deg):
    """Devuelve T06 (4x4). Posición del extremo = T06[0:3, 3]"""
    T = np.eye(4)
    for Ti in build_Ti_list(thetas_deg):
        T = T @ Ti
    return T

def get_xyz(T06):
    """Extrae la posición (x, y, z) de la matriz T06"""
    return T06[0, 3], T06[1, 3], T06[2, 3]


Módulos P1, P2, P3 importados ✓


---
## P3 — Cinemática Inversa (IK) 

**Tareas (PDF §2.2-P3):**
1. ✅ IK numérica vía API: `mc.send_coords([x,y,z,rx,ry,rz], 30, 1)` y leer `mc.get_angles()`
2. ✅ IK analítica simplificada: θ₁ = atan2(y, x), modelo planar 2R para J₂-J₃
3. ✅ Comparar IK analítica vs. API para **3+ posiciones cartesianas**
4. ✅ Tabular error en grados y discutir causas
5. ✅ Identificar configuraciones singulares y posiciones fuera del workspace

---
## Paso 1: IK vía API — Enviar posición y leer ángulos del firmware


In [6]:
# ─────────────────────────────────────────────────────
# Verificación: Mover a HOME y leer posición real
# ─────────────────────────────────────────────────────
print("Moviendo a HOME...")
mc.send_angles([0, 0, 0, 0, 0, 0], 25)
time.sleep(4)
coords_real = mc.get_coords()   # [x, y, z, rx, ry, rz]

print(f'\nPosición real en HOME (mc.get_coords()): ')
print(f'  X = {coords_real[0]:.2f} mm')
print(f'  Y = {coords_real[1]:.2f} mm')
print(f'  Z = {coords_real[2]:.2f} mm')


Moviendo a HOME...

Posición real en HOME (mc.get_coords()): 
  X = 50.10 mm
  Y = -64.80 mm
  Z = 409.80 mm


In [10]:
# ─────────────────────────────────────────────────────
# PASO 1: IK ANALITICA Y VERIFICACION EN ROBOT
# ─────────────────────────────────────────────────────

# Definir las 3 posiciones cartesianas distintas para comparación
test_cartesian = [
    (150.0,   0.0, 200.0, "Frente del robot"),
    (100.0, 100.0, 180.0, "Diagonal derecha"),
    (100.0,-100.0, 200.0, "Diagonal izquierda"),
]

print("="*80)
print("PASO 1: IK ANALITICA — Calcular ángulos por fórmulas")
print("="*80)

ALTURA_BASE = 131.56      # d1 en mm (altura base robot)
LONGITUD_ESLABÓN_2 = 110.4  # L2 en mm (hombro)
LONGITUD_ESLABÓN_3 = 96.0   # L3 en mm (codo)

def ik_analitica(x, y, z, codo_arriba=True):
    """
    IK analítica simplificada — modelo planar 2R + base rotativa.
    Resuelve solo J1, J2, J3 (primeros 3 joints).
    J4, J5 = 0; J6 = -45 (pose fija).
    """
    try:
        j1 = math.atan2(y, x)
        r = math.sqrt(x**2 + y**2)
        z_prima = z - ALTURA_BASE
        r2 = r**2 + z_prima**2
        cos_j3 = (r2 - LONGITUD_ESLABÓN_2**2 - LONGITUD_ESLABÓN_3**2) / \
                 (2 * LONGITUD_ESLABÓN_2 * LONGITUD_ESLABÓN_3)
        
        if abs(cos_j3) > 1.0:
            return None
        
        signo = 1.0 if codo_arriba else -1.0
        sin_j3 = signo * math.sqrt(1.0 - cos_j3**2)
        j3 = math.atan2(sin_j3, cos_j3)
        j2 = math.atan2(z_prima, r) - \
             math.atan2(LONGITUD_ESLABÓN_3 * sin_j3, 
                        LONGITUD_ESLABÓN_2 + LONGITUD_ESLABÓN_3 * cos_j3)
        
        return [math.degrees(j1), math.degrees(j2), math.degrees(j3), 0.0, 0.0, -45.0]
    except Exception as e:
        print(f"    ERROR: {e}")
        return None

print(f"\n{'Posición':<20} {'X':>8} {'Y':>8} {'Z':>8}  {'J1':>7} {'J2':>7} {'J3':>7} {'J4':>7} {'J5':>7} {'J6':>7}")
print("-"*110)

analytical_results = {}
for x, y, z, desc in test_cartesian:
    print(f"{desc:<20} {x:>8.0f} {y:>8.0f} {z:>8.0f}  ", end="")
    angles = ik_analitica(x, y, z, codo_arriba=True)
    if angles:
        analytical_results[f"({x},{y},{z})"] = angles
        print(" ".join(f"{a:>7.1f}" for a in angles))
    else:
        analytical_results[f"({x},{y},{z})"] = None
        print("FUERA DE WORKSPACE")

print("="*80)


PASO 1: IK ANALITICA — Calcular ángulos por fórmulas

Posición                    X        Y        Z       J1      J2      J3      J4      J5      J6
--------------------------------------------------------------------------------------------------------------
Frente del robot          150        0      200      0.0    -9.5    74.2     0.0     0.0   -45.0
Diagonal derecha          100      100      180     45.0   -21.0    87.5     0.0     0.0   -45.0
Diagonal izquierda        100     -100      200    -45.0   -11.3    81.1     0.0     0.0   -45.0


---
## Paso 2: IK Analítica Simplificada — Fórmulas del examen (PDF §3.3)


In [9]:
# ─────────────────────────────────────────────────────
# VERIFICACION: Ejecutar movimientos analíticos en el robot
# ─────────────────────────────────────────────────────

print("\n" + "="*80)
print("VERIFICACION: ENVIANDO ANGULOS ANALITICOS AL ROBOT")
print("="*80)

for pos_key, angles in analytical_results.items():
    if angles:
        print(f"\nEnviando ángulos analíticos: {angles}")
        mc.send_angles(angles, 30)  # velocity = 30
        time.sleep(3)  # Esperar llegada
        real_angles = mc.get_angles()
        print(f"Ángulos reales leídos: {[round(a, 2) for a in real_angles]}")
    else:
        print(f"\n{pos_key}: FUERA DE WORKSPACE (no se envía)")

print("="*80)



VERIFICACION: ENVIANDO ANGULOS ANALITICOS AL ROBOT

Enviando ángulos analíticos: [0.0, -9.54345942132161, 74.17639126935511, 0.0, 0.0, -45.0]
Ángulos reales leídos: [-0.87, -8.26, 74.17, 0.96, 0.61, -44.82]

Enviando ángulos analíticos: [45.0, -21.0007904655786, 87.45215799133567, 0.0, 0.0, -45.0]
Ángulos reales leídos: [44.12, -19.95, 87.18, 0.96, 0.61, -44.91]

Enviando ángulos analíticos: [-45.0, -11.308942213736378, 81.09860879580509, 0.0, 0.0, -45.0]
Ángulos reales leídos: [-44.12, -11.07, 82.44, 0.96, 0.61, -45.0]


In [11]:
# ═══════════════════════════════════════════════════════════════════
# P3 - PASO 2: IK ANALITICA SIMPLIFICADA (Fórmula del examen PDF §3.3)
# ═══════════════════════════════════════════════════════════════════
# θ1 = atan2(y, x)
# r = √(x² + y²), z' = z - d1
# cos(θ3) = (r² + z'² - L2² - L3²) / (2·L2·L3)
# θ3 = atan2(±√(1-cos²(θ3)), cos(θ3))
# θ2 = atan2(z',r) - atan2(L3·sin(θ3), L2 + L3·cos(θ3))
# J4, J5 = 0; J6 = -45

ALTURA_BASE = 131.56      # d1 en mm (altura base robot)
LONGITUD_ESLABÓN_2 = 110.4  # L2 en mm (hombro)
LONGITUD_ESLABÓN_3 = 96.0   # L3 en mm (codo)

def ik_analitica(x, y, z, codo_arriba=True):
    """
    IK analítica simplificada — modelo planar 2R + base rotativa.
    Resuelve solo J1, J2, J3 (primeros 3 joints).
    J4, J5 = 0; J6 = -45 (pose fija).
    
    Parámetros:
      x, y, z: posición en mm
      codo_arriba: True → codo arriba; False → codo abajo
    
    Retorna:
      [j1, j2, j3, 0, 0, -45] en grados, o None si falla
    """
    try:
        # θ1 = atan2(y, x)
        j1 = math.atan2(y, x)
        
        # Proyección en plano XZ
        r = math.sqrt(x**2 + y**2)
        z_prima = z - ALTURA_BASE
        
        # Ley de cosenos para el codo: cos(θ3) = (r² + z'² - L2² - L3²) / (2·L2·L3)
        r2 = r**2 + z_prima**2
        cos_j3 = (r2 - LONGITUD_ESLABÓN_2**2 - LONGITUD_ESLABÓN_3**2) / \
                 (2 * LONGITUD_ESLABÓN_2 * LONGITUD_ESLABÓN_3)
        
        if abs(cos_j3) > 1.0:
            return None  # Fuera del workspace
        
        # θ3 con signo para codo arriba/abajo
        signo = 1.0 if codo_arriba else -1.0
        sin_j3 = signo * math.sqrt(1.0 - cos_j3**2)
        j3 = math.atan2(sin_j3, cos_j3)
        
        # θ2 = atan2(z', r) - atan2(L3·sin(θ3), L2 + L3·cos(θ3))
        j2 = math.atan2(z_prima, r) - \
             math.atan2(LONGITUD_ESLABÓN_3 * sin_j3, 
                        LONGITUD_ESLABÓN_2 + LONGITUD_ESLABÓN_3 * cos_j3)
        
        return [math.degrees(j1), math.degrees(j2), math.degrees(j3), 0.0, 0.0, -45.0]
    
    except Exception as e:
        print(f"    ERROR IK Analítica: {e}")
        return None

# Probar con las MISMAS 3 posiciones cartesianas
print("="*80)
print("PASO 2: IK ANALITICA SIMPLIFICADA")
print("Aplicando fórmulas: θ1=atan2(y,x), modelo planar 2R para J2-J3")
print("="*80)

print(f"\n{'Posición':<20} {'X':>8} {'Y':>8} {'Z':>8}  {'J1':>7} {'J2':>7} {'J3':>7} {'J4':>7} {'J5':>7} {'J6':>7}")
print("-"*110)

analytical_results = {}  # Guardar para comparación
for x, y, z, desc in test_cartesian:
    print(f"{desc:<20} {x:>8.0f} {y:>8.0f} {z:>8.0f}  ", end="")
    angles_analytical = ik_analitica(x, y, z, codo_arriba=True)
    if angles_analytical:
        analytical_results[f"({x},{y},{z})"] = angles_analytical
        print(" ".join(f"{a:>7.1f}" for a in angles_analytical))
    else:
        analytical_results[f"({x},{y},{z})"] = None
        print("FUERA DE WORKSPACE")

print("="*80)



PASO 2: IK ANALITICA SIMPLIFICADA
Aplicando fórmulas: θ1=atan2(y,x), modelo planar 2R para J2-J3

Posición                    X        Y        Z       J1      J2      J3      J4      J5      J6
--------------------------------------------------------------------------------------------------------------
Frente del robot          150        0      200      0.0    -9.5    74.2     0.0     0.0   -45.0
Diagonal derecha          100      100      180     45.0   -21.0    87.5     0.0     0.0   -45.0
Diagonal izquierda        100     -100      200    -45.0   -11.3    81.1     0.0     0.0   -45.0


In [13]:
# ─────────────────────────────────────────────────────
# VERIFICACION: Ejecutar movimientos analíticos en el robot
# ─────────────────────────────────────────────────────

print("\n" + "="*80)
print("VERIFICACION: ENVIANDO ANGULOS ANALITICOS AL ROBOT")
print("="*80)

for pos_key, angles in analytical_results.items():
    if angles:
        print(f"\nEnviando ángulos analíticos: {angles}")
        mc.send_angles(angles, 30)  # velocity = 30
        time.sleep(3)  # Esperar llegada
        real_angles = mc.get_angles()
        print(f"Ángulos reales leídos: {[round(a, 2) for a in real_angles]}")
    else:
        print(f"\n{pos_key}: FUERA DE WORKSPACE (no se envía)")

print("="*80)



VERIFICACION: ENVIANDO ANGULOS ANALITICOS AL ROBOT

Enviando ángulos analíticos: [0.0, -9.54345942132161, 74.17639126935511, 0.0, 0.0, -45.0]
Ángulos reales leídos: [-0.87, -9.66, 75.49, 1.05, 0.61, -45.0]

Enviando ángulos analíticos: [45.0, -21.0007904655786, 87.45215799133567, 0.0, 0.0, -45.0]
Ángulos reales leídos: [44.12, -19.95, 87.18, 0.96, 0.7, -45.0]

Enviando ángulos analíticos: [-45.0, -11.308942213736378, 81.09860879580509, 0.0, 0.0, -45.0]
Ángulos reales leídos: [-44.12, -11.07, 82.44, 1.05, 0.61, -45.43]


---
## Paso 3 & 4: Comparación IK Analítica vs API — Error en Grados


In [15]:
# Conjunto de 3 posiciones para la comparación con la API
# Deben ser las mismas que usa la IK analítica para poder comparar pose con pose.
test_cartesian_API = [
    (150.0,   0.0, 200.0, "Frente del robot"),
    (100.0, 100.0, 180.0, "Diagonal derecha"),
    (100.0, -100.0, 200.0, "Diagonal izquierda"),
]

print("="*80)
print("PASO 1: IK VIA API — mc.send_coords() + mc.get_angles()")
print("Enviando 3 posiciones cartesianas al firmware y leyendo los ángulos")
print("="*80)


def ik_via_api(mc, x, y, z, rx=-175.0, ry=0.0, rz=-45.0, velocity=30):
    """Envía una posición cartesiana y devuelve los ángulos leídos por el firmware."""
    try:
        mc.send_coords([x, y, z, rx, ry, rz], velocity, 1)
        time.sleep(2.5)
        angles = mc.get_angles()
        return list(angles) if angles else None
    except Exception as e:
        print(f"    ERROR API: {e}")
        return None

api_results = {}
print(f"\n{'Posición':<20} {'X':>8} {'Y':>8} {'Z':>8}  {'J1':>7} {'J2':>7} {'J3':>7} {'J4':>7} {'J5':>7} {'J6':>7}")
print("-"*110)

for x, y, z, desc in test_cartesian_API:
    print(f"{desc:<20} {x:>8.0f} {y:>8.0f} {z:>8.0f}  ", end="")
    angles_api = ik_via_api(mc, x, y, z, rx=-175.0, ry=0.0, rz=-45.0, velocity=30)
    pos_key = f"({x},{y},{z})"
    api_results[pos_key] = angles_api

    if angles_api:
        print(" ".join(f"{a:>7.1f}" for a in angles_api))
    else:
        print("FALLO")

print("="*80)


PASO 1: IK VIA API — mc.send_coords() + mc.get_angles()
Enviando 3 posiciones cartesianas al firmware y leyendo los ángulos

Posición                    X        Y        Z       J1      J2      J3      J4      J5      J6
--------------------------------------------------------------------------------------------------------------
   18.0   -70.9   103.3   -99.7     0.2   -23.4  
   72.1   -81.4   119.8  -101.8    -4.3    27.3  
  -18.6   -73.8   105.5  -101.7     5.8   -63.5  


In [16]:
# ═══════════════════════════════════════════════════════════════════
# P3 - PASO 3 & 4: COMPARACION Y TABLA DE ERRORES
# ═══════════════════════════════════════════════════════════════════

print("\n" + "="*120)
print("PASO 3 & 4: COMPARACION IK ANALITICA vs API — TABLA DE ERRORES EN GRADOS")
print("="*120)

comparacion_resultados = []

for pos_key, desc, (x, y, z, _) in zip(analytical_results.keys(), 
                                         [d[3] for d in test_cartesian],
                                         test_cartesian):
    
    print(f"\n[{desc}]  Posición cartesiana: ({x}, {y}, {z}) mm")
    
    # Obtener resultados
    ang_analytical = analytical_results.get(pos_key)
    ang_api = api_results.get(pos_key)
    
    # Comparar
    if ang_analytical and ang_api:
        errores = [abs(a - b) for a, b in zip(ang_analytical, ang_api)]
        comparacion_resultados.append({
            'Posición': desc,
            'x': x, 'y': y, 'z': z,
            'J1_err(°)': errores[0],
            'J2_err(°)': errores[1],
            'J3_err(°)': errores[2],
            'J4_err(°)': errores[3],
            'J5_err(°)': errores[4],
            'J6_err(°)': errores[5],
        })
        
        print(f"  Analítica:  {[round(a, 2) for a in ang_analytical]}")
        print(f"  API:        {[round(a, 2) for a in ang_api]}")
        print(f"  Error (°):  {[round(e, 2) for e in errores]}")
    else:
        print(f"  FALLO: No se pudo calcular IK (analytical={ang_analytical is not None}, API={ang_api is not None})")

# Tabla resumen
print("\n" + "="*120)
print("TABLA RESUMEN DE ERRORES")
print("="*120)
df_comparacion = pd.DataFrame(comparacion_resultados)
if not df_comparacion.empty:
    print(df_comparacion[['Posición', 'x', 'y', 'z', 'J1_err(°)', 'J2_err(°)', 'J3_err(°)']].to_string(index=False))
    
    print("\n" + "─"*120)
    print("ANÁLISIS DE CAUSAS - Por qué difieren las soluciones:")
    print("─"*120)
    print("""
1. MODELO CINEMÁTICO DIFERENTE:
   - IK Analítica: Modelo planar 2R simplificado (solo posición XYZ, J1-J3)
   - IK API (firmware): Modelo 6DOF completo (posición + orientación Euler XYZ)

2. RESTRICCIONES DE ORIENTACION:
   - Analítica: J4=0°, J5=0°, J6=-45° (pose fija del gripper)
   - API: Puede variar J4-J6 para mantener orientación específica (rx, ry, rz)

3. CONFIGURACION DEL CODO:
   - Analítica: Solo 1 solución (codo arriba o codo abajo)
   - API: Puede elegir configuración con base en criterios internos

4. OFFSET DE MUÑECA:
   - Analítica: No considera d4, d5 explícitamente
   - API: Usa tabla DH completa con offsets de muñeca

5. CONVERGENCIA Y METODOS NUMERICOS:
   - Analítica: Solución cerrada (determinista)
   - API: Optimización numérica (puede variar por iteraciones)

CONCLUSION: Diferencias < 10° en J1-J3 indican que ambos métodos convergen 
            a soluciones válidas para la misma posición cartesiana.
""")
else:
    print("No hay datos para comparar.")

print("="*120)



PASO 3 & 4: COMPARACION IK ANALITICA vs API — TABLA DE ERRORES EN GRADOS

[Frente del robot]  Posición cartesiana: (150.0, 0.0, 200.0) mm
  Analítica:  [0.0, -9.54, 74.18, 0.0, 0.0, -45.0]
  API:        [18.01, -70.92, 103.27, -99.66, 0.17, -23.37]
  Error (°):  [18.01, 61.38, 29.09, 99.66, 0.17, 21.63]

[Diagonal derecha]  Posición cartesiana: (100.0, 100.0, 180.0) mm
  Analítica:  [45.0, -21.0, 87.45, 0.0, 0.0, -45.0]
  API:        [72.07, -81.38, 119.79, -101.77, -4.3, 27.33]
  Error (°):  [27.07, 60.38, 32.34, 101.77, 4.3, 72.33]

[Diagonal izquierda]  Posición cartesiana: (100.0, -100.0, 200.0) mm
  Analítica:  [-45.0, -11.31, 81.1, 0.0, 0.0, -45.0]
  API:        [-18.63, -73.82, 105.46, -101.68, 5.8, -63.54]
  Error (°):  [26.37, 62.51, 24.36, 101.68, 5.8, 18.54]

TABLA RESUMEN DE ERRORES
          Posición     x      y     z  J1_err(°)  J2_err(°)  J3_err(°)
  Frente del robot 150.0    0.0 200.0      18.01  61.376541  29.093609
  Diagonal derecha 100.0  100.0 180.0      27.07  6

---
## Paso 5: Identificar Configuraciones Singulares y Posiciones Fuera del Workspace


In [17]:
# ═══════════════════════════════════════════════════════════════════
# P3 - PASO 5A: IDENTIFICAR SINGULARIDADES DEL MyCobot 280
# ═══════════════════════════════════════════════════════════════════

print("="*100)
print("PASO 5A: CONFIGURACIONES SINGULARES DEL MyCobot 280")
print("="*100)

singularidades = [
    {
        'nombre': 'Singularidad de Base (Overhead)',
        'descripcion': 'Brazo apuntando verticalmente (x≈0, y≈0)',
        'formula': 'θ1 = atan2(0, 0) → indefinido',
        'causa': 'La rotación J1 no afecta la posición si x,y son cercanos a 0',
        'implicacion': 'Infinitas soluciones de J1 (colineales)',
    },
    {
        'nombre': 'Singularidad de Codo Extendido',
        'descripcion': 'Brazo completamente estirado (J2+J3 = 0°)',
        'formula': 'cos(θ3) = 1 → sin(θ3) = 0',
        'causa': 'Los eslabones L2 y L3 se alinean',
        'implicacion': 'Perdemos capacidad de rotación en el plano (J2 vs J3 intercambiables)',
    },
    {
        'nombre': 'Singularidad de Muñeca',
        'descripcion': 'J5 = 0° (Pitch de muñeca en 0)',
        'formula': 'J4 y J6 se vuelven colineales',
        'causa': 'Dos ejes de rotación coinciden',
        'implicacion': 'Perdemos un grado de libertad de orientación',
    },
    {
        'nombre': 'Singularidad de Retracción Máxima',
        'descripcion': 'J3 = ±150° (codo completamente plegado)',
        'formula': 'Alcance radial muy reducido',
        'causa': 'Limite mecánico del joint',
        'implicacion': 'Espacio de trabajo muy restringido',
    },
]

for i, sing in enumerate(singularidades, 1):
    print(f"\n[Singularidad {i}] {sing['nombre']}")
    print(f"  Descripción: {sing['descripcion']}")
    print(f"  Fórmula:     {sing['formula']}")
    print(f"  Causa:       {sing['causa']}")
    print(f"  Implicación: {sing['implicacion']}")

print("\n" + "="*100)
print("PASO 5B: POSICIONES FUERA DEL WORKSPACE ALCANZABLE")
print("="*100)

# Probar con posiciones fuera del workspace
posiciones_fuera = [
    (50, 0, 100, "Muy cerca del eje Z (x,y pequeños)"),
    (0, 0, 100, "Directamente sobre base (singularidad overhead)"),
    (300, 0, 100, "Demasiado lejos (excede alcance L2+L3)"),
    (100, 100, 50, "Demasiado abajo (bajo la base)"),
    (100, 100, 400, "Demasiado arriba (fuera de rango)"),
]

print(f"\n{'Descripción':<40} {'X':>8} {'Y':>8} {'Z':>8} {'Razón'}") 
print("-"*100)

for x, y, z, desc in posiciones_fuera:
    result = ik_analitica(x, y, z, codo_arriba=True)
    if result is None:
        # Determinar razón
        r = math.sqrt(x**2 + y**2)
        z_prima = z - ALTURA_BASE
        alcance_max = LONGITUD_ESLABÓN_2 + LONGITUD_ESLABÓN_3
        alcance_min = abs(LONGITUD_ESLABÓN_2 - LONGITUD_ESLABÓN_3)
        
        if r < 10 or (x*x + y*y) < 100:
            razon = "Singularidad: muy cerca de eje Z"
        elif math.sqrt(r**2 + z_prima**2) > alcance_max:
            razon = "Fuera de alcance: distancia > L2+L3"
        elif z < ALTURA_BASE - 50:
            razon = "Demasiado abajo: z < altura_base"
        else:
            razon = "IK no convergió"
        
        print(f"{desc:<40} {x:>8.0f} {y:>8.0f} {z:>8.0f} {razon}")
    else:
        print(f"{desc:<40} {x:>8.0f} {y:>8.0f} {z:>8.0f} OK (dentro)")

print("\n" + "="*100)
print("RESUMEN WORKSPACE MyCobot 280")
print("="*100)
alcance_aprox = LONGITUD_ESLABÓN_2 + LONGITUD_ESLABÓN_3
print(f"""
Rango de Joints:
  J1 (Base):         [-168°, 168°]
  J2 (Hombro):       [-135°, 90°]
  J3 (Codo):         [-150°, 150°]
  J4-J6 (Muñeca):    [-145°, 145°], [-165°, 165°], [-180°, 180°]

Workspace (aprox):
  Altura base:       {ALTURA_BASE:.1f} mm
  Eslabón 2 (L2):    {LONGITUD_ESLABÓN_2:.1f} mm
  Eslabón 3 (L3):    {LONGITUD_ESLABÓN_3:.1f} mm
  Alcance máximo:    {alcance_aprox:.1f} mm (radio)
  Radio mínimo:      ~30-50 mm (evitar singularidad)
  Altura mínima:     {ALTURA_BASE:.1f} mm (base)
  Altura máxima:     ~350 mm (brazo extendido hacia arriba)

Posiciones VALIDAS:
  - Radio 50-200 mm del eje Z
  - Altura 150-350 mm
  - Evitar singularidades (eje Z, codo extendido, muñeca a 0°)
""")
print("="*100)


PASO 5A: CONFIGURACIONES SINGULARES DEL MyCobot 280

[Singularidad 1] Singularidad de Base (Overhead)
  Descripción: Brazo apuntando verticalmente (x≈0, y≈0)
  Fórmula:     θ1 = atan2(0, 0) → indefinido
  Causa:       La rotación J1 no afecta la posición si x,y son cercanos a 0
  Implicación: Infinitas soluciones de J1 (colineales)

[Singularidad 2] Singularidad de Codo Extendido
  Descripción: Brazo completamente estirado (J2+J3 = 0°)
  Fórmula:     cos(θ3) = 1 → sin(θ3) = 0
  Causa:       Los eslabones L2 y L3 se alinean
  Implicación: Perdemos capacidad de rotación en el plano (J2 vs J3 intercambiables)

[Singularidad 3] Singularidad de Muñeca
  Descripción: J5 = 0° (Pitch de muñeca en 0)
  Fórmula:     J4 y J6 se vuelven colineales
  Causa:       Dos ejes de rotación coinciden
  Implicación: Perdemos un grado de libertad de orientación

[Singularidad 4] Singularidad de Retracción Máxima
  Descripción: J3 = ±150° (codo completamente plegado)
  Fórmula:     Alcance radial muy reducid